<a href="https://colab.research.google.com/github/pjastr-uwm/fakultet_io_2026/blob/main/lecture4/korpus_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Jak przygotować korpus do dalszych analiz NLP?

> **Notatka praktyczna** — pozyskiwanie danych z różnych formatów plików, budowa korpusu tekstowego oraz przegląd popularnych datasetów dla języka angielskiego i polskiego.

---

## Spis treści

1. [Instalacja zależności](#1-instalacja-zależności)
2. [Pozyskiwanie tekstu z plików TXT](#2-pozyskiwanie-tekstu-z-plików-txt)
3. [Pozyskiwanie tekstu z plików CSV / TSV](#3-pozyskiwanie-tekstu-z-plików-csv--tsv)
4. [Pozyskiwanie tekstu z plików Word (.docx)](#4-pozyskiwanie-tekstu-z-plików-word-docx)
5. [Pozyskiwanie tekstu z plików PDF](#5-pozyskiwanie-tekstu-z-plików-pdf)
6. [Budowa ujednoliconego korpusu](#6-budowa-ujednoliconego-korpusu)
7. [Podstawowe czyszczenie tekstu](#7-podstawowe-czyszczenie-tekstu)
8. [Popularne korpusy i datasety — język angielski](#8-popularne-korpusy-i-datasety--język-angielski)
9. [Popularne korpusy i datasety — język polski](#9-popularne-korpusy-i-datasety--język-polski)
10. [Podsumowanie](#10-podsumowanie)

---
## 1. Instalacja zależności

Poniższa komórka instaluje wszystkie potrzebne biblioteki. W Google Colab większość z nich jest już dostępna — instalujemy tylko brakujące.

In [1]:
# Instalacja bibliotek (Google Colab)
!pip install -q python-docx pdfplumber PyPDF2 datasets nltk pandas openpyxl chardet
!pip install -q reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.5 MB/s eta 0:00:00


In [2]:
import os
import re
import glob
import chardet
import pandas as pd

# Ustawienie wyświetlania
pd.set_option('display.max_colwidth', 120)

print("Podstawowe biblioteki załadowane.")

Podstawowe biblioteki załadowane.


---
## 2. Pozyskiwanie tekstu z plików TXT

Pliki `.txt` to najprostszy format. Kluczowe kwestie:
- **Kodowanie** — pliki w języku polskim mogą być zapisane w UTF-8, CP-1250 (Windows) lub ISO-8859-2. Warto je automatycznie wykrywać.
- **Struktura** — zwykle płaska; ewentualnie podział na akapity (puste linie).

In [3]:
# --- Tworzenie przykładowych plików TXT do demonstracji ---
os.makedirs("dane_przykladowe", exist_ok=True)

tekst_pl = """Przetwarzanie języka naturalnego (NLP) to dziedzina sztucznej inteligencji,
która zajmuje się interakcją między komputerami a językiem ludzkim.

Współczesne metody NLP opierają się na modelach uczenia głębokiego,
takich jak transformery (np. BERT, GPT).

Budowa dobrego korpusu tekstowego to kluczowy pierwszy krok
w każdym projekcie NLP."""

# Zapis w UTF-8
with open("dane_przykladowe/tekst_utf8.txt", "w", encoding="utf-8") as f:
    f.write(tekst_pl)

# Zapis w CP-1250 (typowe kodowanie Windows dla polskiego)
with open("dane_przykladowe/tekst_cp1250.txt", "w", encoding="cp1250") as f:
    f.write(tekst_pl)

print("Pliki TXT utworzone.")

Pliki TXT utworzone.


In [4]:
def wczytaj_txt(sciezka, kodowanie=None):
    """
    Wczytuje plik TXT. Jeśli kodowanie nie jest podane,
    próbuje je automatycznie wykryć za pomocą biblioteki chardet.
    """
    if kodowanie is None:
        with open(sciezka, "rb") as f:
            surowe = f.read()
        wykryte = chardet.detect(surowe)
        kodowanie = wykryte["encoding"]
        print(f"  Wykryte kodowanie: {kodowanie} (pewność: {wykryte['confidence']:.0%})")

    with open(sciezka, "r", encoding=kodowanie) as f:
        tekst = f.read()
    return tekst


# --- Odczyt pliku UTF-8 ---
print("== tekst_utf8.txt ==")
tekst1 = wczytaj_txt("dane_przykladowe/tekst_utf8.txt")
print(tekst1[:200])

print("\n== tekst_cp1250.txt ==")
tekst2 = wczytaj_txt("dane_przykladowe/tekst_cp1250.txt")
print(tekst2[:200])

== tekst_utf8.txt ==
  Wykryte kodowanie: utf-8 (pewność: 99%)
Przetwarzanie języka naturalnego (NLP) to dziedzina sztucznej inteligencji,
która zajmuje się interakcją między komputerami a językiem ludzkim.

Współczesne metody NLP opierają się na modelach uczenia

== tekst_cp1250.txt ==
  Wykryte kodowanie: ISO-8859-1 (pewność: 73%)
Przetwarzanie jêzyka naturalnego (NLP) to dziedzina sztucznej inteligencji,
która zajmuje siê interakcj¹ miêdzy komputerami a jêzykiem ludzkim.

Wspó³czesne metody NLP opieraj¹ siê na modelach uczenia


In [5]:
def wczytaj_wszystkie_txt(katalog, wzorzec="*.txt"):
    """Wczytuje wszystkie pliki TXT z katalogu do listy słowników."""
    dokumenty = []
    for sciezka in sorted(glob.glob(os.path.join(katalog, wzorzec))):
        tekst = wczytaj_txt(sciezka)
        dokumenty.append({
            "plik": os.path.basename(sciezka),
            "tekst": tekst,
            "liczba_znakow": len(tekst),
        })
    return dokumenty

docs_txt = wczytaj_wszystkie_txt("dane_przykladowe")
df_txt = pd.DataFrame(docs_txt)
print(f"Wczytano {len(df_txt)} plików TXT:")
df_txt[["plik", "liczba_znakow"]]

  Wykryte kodowanie: ISO-8859-1 (pewność: 73%)
  Wykryte kodowanie: utf-8 (pewność: 99%)
Wczytano 2 plików TXT:


,plik,liczba_znakow
0,tekst_cp1250.txt,338
1,tekst_utf8.txt,338


---
## 3. Pozyskiwanie tekstu z plików CSV / TSV

Pliki CSV/TSV często zawierają dane strukturalne, np.:
- recenzje z kolumnami `tekst`, `ocena`,
- artykuły z kolumnami `tytul`, `tresc`, `data`.

Pandas świetnie sobie z nimi radzi. Trzeba zwrócić uwagę na separator i kodowanie.

In [6]:
# --- Tworzenie przykładowego CSV ---
dane_csv = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "tekst": [
        "Film był naprawdę świetny, polecam każdemu!",
        "Tragiczna gra aktorska i nudna fabuła.",
        "Nieźle, ale mogło być lepiej. Efekty specjalne OK.",
        "Arcydzieło kina — dawno nie widziałem czegoś tak poruszającego."
    ],
    "ocena": [5, 1, 3, 5]
})

dane_csv.to_csv("dane_przykladowe/recenzje.csv", index=False, encoding="utf-8")
# Wariant TSV (tab-separated)
dane_csv.to_csv("dane_przykladowe/recenzje.tsv", index=False, encoding="utf-8", sep="\t")

print("Pliki CSV / TSV utworzone.")

Pliki CSV / TSV utworzone.


In [7]:
# --- Odczyt CSV ---
df_csv = pd.read_csv("dane_przykladowe/recenzje.csv")
print("Kolumny:", list(df_csv.columns))
print(f"Wiersze: {len(df_csv)}\n")
df_csv

Kolumny: ['id', 'tekst', 'ocena']
Wiersze: 4



,id,tekst,ocena
0,1,"Film był naprawdę świetny, polecam każdemu!",5
1,2,Tragiczna gra aktorska i nudna fabuła.,1
2,3,"Nieźle, ale mogło być lepiej. Efekty specjalne OK.",3
3,4,Arcydzieło kina — dawno nie widziałem czegoś tak poruszającego.,5


In [8]:
# --- Odczyt TSV ---
df_tsv = pd.read_csv("dane_przykladowe/recenzje.tsv", sep="\t")

# Wyciągnięcie samych tekstów do listy (typowe przy budowie korpusu)
korpus_z_csv = df_tsv["tekst"].tolist()
for i, t in enumerate(korpus_z_csv):
    print(f"[{i}] {t}")

[0] Film był naprawdę świetny, polecam każdemu!
[1] Tragiczna gra aktorska i nudna fabuła.
[2] Nieźle, ale mogło być lepiej. Efekty specjalne OK.
[3] Arcydzieło kina — dawno nie widziałem czegoś tak poruszającego.


### Wskazówki dotyczące CSV
- Użyj parametru `encoding` (np. `"utf-8"`, `"cp1250"`, `"latin2"`) jeśli polskie znaki się „sypią".
- Duże pliki wczytuj partiami: `pd.read_csv(..., chunksize=10000)`.
- Jeśli plik ma niestandardowy separator, użyj `sep="|"` itp.

---
## 4. Pozyskiwanie tekstu z plików Word (.docx)

Biblioteka **python-docx** pozwala wyciągać:
- tekst akapitów,
- tekst z tabel,
- metadane dokumentu (autor, tytuł, data).

> ⚠️ Obsługuje tylko format `.docx` (Office 2007+). Dla starszego `.doc` potrzebne są inne narzędzia (np. `antiword`, `libreoffice --convert-to`).

In [9]:
from docx import Document
from docx.shared import Pt

# --- Tworzenie przykładowego pliku DOCX ---
doc = Document()
doc.add_heading("Raport z analizy NLP", level=1)
doc.add_paragraph(
    "Niniejszy raport przedstawia wyniki analizy korpusu tekstów "
    "w języku polskim zebranego z różnych źródeł internetowych."
)

doc.add_heading("Metodologia", level=2)
doc.add_paragraph(
    "Do budowy korpusu wykorzystano artykuły prasowe, posty z forów "
    "oraz recenzje produktów. Łączna objętość to ok. 2 mln tokenów."
)

# Dodajmy tabelę
doc.add_heading("Statystyki", level=2)
tabela = doc.add_table(rows=4, cols=3)
tabela.style = "Table Grid"
naglowki = ["Źródło", "Liczba dokumentów", "Liczba tokenów"]
for i, h in enumerate(naglowki):
    tabela.rows[0].cells[i].text = h
dane_tab = [
    ("Artykuły prasowe", "1 200", "980 000"),
    ("Fora internetowe", "3 500", "750 000"),
    ("Recenzje", "5 000", "270 000"),
]
for r, (a, b, c) in enumerate(dane_tab, start=1):
    tabela.rows[r].cells[0].text = a
    tabela.rows[r].cells[1].text = b
    tabela.rows[r].cells[2].text = c

doc.save("dane_przykladowe/raport_nlp.docx")
print("Plik DOCX utworzony.")

Plik DOCX utworzony.


In [10]:
def wczytaj_docx(sciezka, z_tabelami=True):
    """
    Wczytuje plik .docx i zwraca słownik z:
    - paragrafami (tekst + styl),
    - tabelami (lista list),
    - metadanymi.
    """
    doc = Document(sciezka)

    # Akapity z informacją o stylu (nagłówek, normalny tekst itp.)
    akapity = []
    for p in doc.paragraphs:
        if p.text.strip():
            akapity.append({
                "tekst": p.text,
                "styl": p.style.name
            })

    # Tabele
    tabele = []
    if z_tabelami:
        for tab in doc.tables:
            wiersze = []
            for wiersz in tab.rows:
                wiersze.append([komorka.text for komorka in wiersz.cells])
            tabele.append(wiersze)

    # Metadane
    meta = {
        "autor": doc.core_properties.author,
        "tytul": doc.core_properties.title,
        "utworzony": str(doc.core_properties.created),
    }

    return {"akapity": akapity, "tabele": tabele, "metadane": meta}


wynik = wczytaj_docx("dane_przykladowe/raport_nlp.docx")

print("=== METADANE ===")
print(wynik["metadane"])

print("\n=== AKAPITY (z zachowaniem struktury) ===")
for a in wynik["akapity"]:
    print(f"  [{a['styl']:20s}]  {a['tekst'][:80]}")

print("\n=== TABELE ===")
for i, tab in enumerate(wynik["tabele"]):
    print(f"Tabela {i+1}:")
    for wiersz in tab:
        print(f"  {wiersz}")

=== METADANE ===
{'autor': 'python-docx', 'tytul': '', 'utworzony': '2013-12-23 23:15:00+00:00'}

=== AKAPITY (z zachowaniem struktury) ===
  [Heading 1           ]  Raport z analizy NLP
  [Normal              ]  Niniejszy raport przedstawia wyniki analizy korpusu tekstów w języku polskim zeb
  [Heading 2           ]  Metodologia
  [Normal              ]  Do budowy korpusu wykorzystano artykuły prasowe, posty z forów oraz recenzje pro
  [Heading 2           ]  Statystyki

=== TABELE ===
Tabela 1:
  ['Źródło', 'Liczba dokumentów', 'Liczba tokenów']
  ['Artykuły prasowe', '1 200', '980 000']
  ['Fora internetowe', '3 500', '750 000']
  ['Recenzje', '5 000', '270 000']


In [11]:
def docx_do_tekstu(sciezka):
    """Wyciąga czysty tekst z pliku DOCX (akapity + tabele)."""
    dane = wczytaj_docx(sciezka)
    fragmenty = [a["tekst"] for a in dane["akapity"]]

    for tab in dane["tabele"]:
        for wiersz in tab:
            fragmenty.append(" | ".join(wiersz))

    return "\n".join(fragmenty)

pelny_tekst = docx_do_tekstu("dane_przykladowe/raport_nlp.docx")
print(pelny_tekst)

Raport z analizy NLP
Niniejszy raport przedstawia wyniki analizy korpusu tekstów w języku polskim zebranego z różnych źródeł internetowych.
Metodologia
Do budowy korpusu wykorzystano artykuły prasowe, posty z forów oraz recenzje produktów. Łączna objętość to ok. 2 mln tokenów.
Statystyki
Źródło | Liczba dokumentów | Liczba tokenów
Artykuły prasowe | 1 200 | 980 000
Fora internetowe | 3 500 | 750 000
Recenzje | 5 000 | 270 000


---
## 5. Pozyskiwanie tekstu z plików PDF

PDF to jeden z trudniejszych formatów do ekstrakcji tekstu, ponieważ:
- tekst może być osadzony jako grafika (skany → wymaga OCR),
- układ wielokolumnowy może być źle interpretowany,
- tabele i stopki/nagłówki mieszają się z treścią.

Używamy dwóch bibliotek:
- **pdfplumber** — świetny do tabel i tekstu z zachowaniem układu,
- **PyPDF2** — lżejsza alternatywa do prostej ekstrakcji.

In [12]:

import pdfplumber
from PyPDF2 import PdfReader as PyPDF2Reader

# --- Tworzenie przykładowego PDF (za pomocą reportlab, jeśli dostępny) ---
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

c = canvas.Canvas("dane_przykladowe/dokument.pdf", pagesize=A4)
width, height = A4

c.setFont("Helvetica-Bold", 18)
c.drawString(50, height - 60, "Analiza tekstu w NLP")

c.setFont("Helvetica", 12)
akapity_pdf = [
    "Przetwarzanie jezyka naturalnego (NLP) laczy lingwistyke z informatyka.",
    "Korpus tekstowy to zbior dokumentow uzywany do trenowania modeli.",
    "Popularne zadania NLP to: klasyfikacja, NER, tlumaczenie, streszczanie.",
    "W jezyku polskim wyzwaniem jest bogata fleksja i wolny szyk zdania.",
]
y = height - 100
for akapit in akapity_pdf:
    c.drawString(50, y, akapit)
    y -= 25

c.save()
print("Plik PDF utworzony (reportlab).")


Plik PDF utworzony (reportlab).


In [13]:
def wczytaj_pdf_pdfplumber(sciezka):
    """
    Wyciąga tekst i tabele z PDF za pomocą pdfplumber.
    Zwraca strukturę: lista stron → tekst + tabele na stronie.
    """
    strony = []
    with pdfplumber.open(sciezka) as pdf:
        for i, strona in enumerate(pdf.pages):
            tekst = strona.extract_text() or ""
            tabele = strona.extract_tables() or []
            strony.append({
                "nr_strony": i + 1,
                "tekst": tekst,
                "tabele": tabele,
            })
    return strony


strony = wczytaj_pdf_pdfplumber("dane_przykladowe/dokument.pdf")
for s in strony:
    print(f"--- Strona {s['nr_strony']} ---")
    print(s["tekst"][:300])
    if s["tabele"]:
        print(f"  Znaleziono {len(s['tabele'])} tabel(e)")

--- Strona 1 ---
Analiza tekstu w NLP
Przetwarzanie jezyka naturalnego (NLP) laczy lingwistyke z informatyka.
Korpus tekstowy to zbior dokumentow uzywany do trenowania modeli.
Popularne zadania NLP to: klasyfikacja, NER, tlumaczenie, streszczanie.
W jezyku polskim wyzwaniem jest bogata fleksja i wolny szyk zdania.


In [14]:
def wczytaj_pdf_pypdf2(sciezka):
    """Prostsza ekstrakcja tekstu za pomocą PyPDF2."""
    reader = PyPDF2Reader(sciezka)
    teksty = []
    for i, strona in enumerate(reader.pages):
        t = strona.extract_text() or ""
        teksty.append(t)
    return "\n\n".join(teksty)

tekst_pdf = wczytaj_pdf_pypdf2("dane_przykladowe/dokument.pdf")
print(tekst_pdf[:500])

Analiza tekstu w NLP
Przetwarzanie jezyka naturalnego (NLP) laczy lingwistyke z informatyka.
Korpus tekstowy to zbior dokumentow uzywany do trenowania modeli.
Popularne zadania NLP to: klasyfikacja, NER, tlumaczenie, streszczanie.
W jezyku polskim wyzwaniem jest bogata fleksja i wolny szyk zdania.



### Wskazówki do ekstrakcji z PDF
- **Skany / obrazy w PDF** → potrzebujesz OCR, np. `pytesseract` + `pdf2image`:
  ```python
  !pip install pytesseract pdf2image
  !apt-get install -y tesseract-ocr tesseract-ocr-pol  # polski model OCR
  ```
- **Wielokolumnowy układ** → `pdfplumber` lepiej radzi sobie z zachowaniem struktury niż PyPDF2.
- **Duże pliki** → przetwarzaj stronę po stronie, nie wczytuj całości do pamięci.

---
## 6. Budowa ujednoliconego korpusu

Po wyciągnięciu tekstu z różnych formatów, warto zebrać wszystko w jedną, ustandaryzowaną strukturę — **DataFrame** lub listę słowników.

In [15]:
def zbuduj_korpus(katalog):
    """
    Skanuje katalog, wczytuje pliki TXT, CSV, DOCX, PDF
    i zwraca ujednolicony DataFrame.
    """
    dokumenty = []

    for plik in sorted(os.listdir(katalog)):
        sciezka = os.path.join(katalog, plik)
        tekst = None
        format_pliku = None

        if plik.endswith(".txt"):
            tekst = wczytaj_txt(sciezka)
            format_pliku = "TXT"

        elif plik.endswith(".csv"):
            df = pd.read_csv(sciezka)
            # Zakładamy, że kolumna z tekstem nazywa się "tekst"
            if "tekst" in df.columns:
                for _, wiersz in df.iterrows():
                    dokumenty.append({
                        "plik": plik,
                        "format": "CSV",
                        "tekst": str(wiersz["tekst"]),
                    })
                continue

        elif plik.endswith(".docx"):
            tekst = docx_do_tekstu(sciezka)
            format_pliku = "DOCX"

        elif plik.endswith(".pdf"):
            tekst = wczytaj_pdf_pypdf2(sciezka)
            format_pliku = "PDF"

        if tekst:
            dokumenty.append({
                "plik": plik,
                "format": format_pliku,
                "tekst": tekst,
            })

    df = pd.DataFrame(dokumenty)
    df["liczba_slow"] = df["tekst"].apply(lambda x: len(x.split()))
    df["liczba_znakow"] = df["tekst"].apply(len)
    return df


korpus = zbuduj_korpus("dane_przykladowe")
print(f"Łączna liczba dokumentów w korpusie: {len(korpus)}")
print(f"Łączna liczba słów: {korpus['liczba_slow'].sum():,}")
print()
korpus[["plik", "format", "liczba_slow", "liczba_znakow"]]

  Wykryte kodowanie: ISO-8859-1 (pewność: 73%)
  Wykryte kodowanie: utf-8 (pewność: 99%)
Łączna liczba dokumentów w korpusie: 8
Łączna liczba słów: 227



,plik,format,liczba_slow,liczba_znakow
0,dokument.pdf,PDF,40,299
1,raport_nlp.docx,DOCX,70,429
2,recenzje.csv,CSV,6,43
3,recenzje.csv,CSV,6,38
4,recenzje.csv,CSV,8,50
5,recenzje.csv,CSV,9,63
6,tekst_cp1250.txt,TXT,44,338
7,tekst_utf8.txt,TXT,44,338


In [16]:
# Zapis korpusu do pliku (do późniejszego wykorzystania)
korpus.to_json("dane_przykladowe/korpus.jsonl", orient="records", lines=True, force_ascii=False)
korpus.to_csv("dane_przykladowe/korpus.csv", index=False)

print("Korpus zapisany jako JSONL i CSV.")
print("\nPrzykładowy rekord JSONL:")
print(open("dane_przykladowe/korpus.jsonl", encoding="utf-8").readline()[:200])

Korpus zapisany jako JSONL i CSV.

Przykładowy rekord JSONL:
{"plik":"dokument.pdf","format":"PDF","tekst":"Analiza tekstu w NLP\nPrzetwarzanie jezyka naturalnego (NLP) laczy lingwistyke z informatyka.\nKorpus tekstowy to zbior dokumentow uzywany do trenowania 


---
## 7. Podstawowe czyszczenie tekstu

Przed analizą NLP korpus zazwyczaj wymaga czyszczenia. Poniżej zestaw typowych operacji.

In [17]:
import unicodedata

def wyczysc_tekst(tekst, male_litery=True, usun_cyfry=False, usun_interpunkcje=False):
    """
    Podstawowe czyszczenie tekstu:
    - normalizacja Unicode (NFC — ważne dla polskich znaków!)
    - usunięcie nadmiarowych białych znaków
    - opcjonalnie: małe litery, usunięcie cyfr, interpunkcji
    """
    # Normalizacja Unicode — zapewnia spójną reprezentację ąćęłńóśźż
    tekst = unicodedata.normalize("NFC", tekst)

    # Usunięcie nadmiarowych białych znaków
    tekst = re.sub(r"\s+", " ", tekst).strip()

    if male_litery:
        tekst = tekst.lower()

    if usun_cyfry:
        tekst = re.sub(r"\d+", "", tekst)

    if usun_interpunkcje:
        tekst = re.sub(r"[^\w\s]", "", tekst)

    return tekst


# Przykład
przyklad = "  Przetwarzanie   Języka Naturalnego\n(NLP)  —  rok 2024!  "
print("Oryginał:  ", repr(przyklad))
print("Wyczyszczony:", wyczysc_tekst(przyklad))
print("Bez interp.: ", wyczysc_tekst(przyklad, usun_interpunkcje=True))

Oryginał:   '  Przetwarzanie   Języka Naturalnego\n(NLP)  —  rok 2024!  '
Wyczyszczony: przetwarzanie języka naturalnego (nlp) — rok 2024!
Bez interp.:  przetwarzanie języka naturalnego nlp  rok 2024


In [18]:
# Zastosowanie czyszczenia do całego korpusu
korpus["tekst_czysty"] = korpus["tekst"].apply(wyczysc_tekst)

print("Przykład przed i po czyszczeniu:")
print("PRZED:", korpus.iloc[0]["tekst"][:100])
print("PO:   ", korpus.iloc[0]["tekst_czysty"][:100])

Przykład przed i po czyszczeniu:
PRZED: Analiza tekstu w NLP
Przetwarzanie jezyka naturalnego (NLP) laczy lingwistyke z informatyka.
Korpus 
PO:    analiza tekstu w nlp przetwarzanie jezyka naturalnego (nlp) laczy lingwistyke z informatyka. korpus 


---
## 8. Popularne korpusy i datasety — język angielski

Poniżej przegląd najczęściej używanych datasetów NLP w języku angielskim, dostępnych przez bibliotekę **Hugging Face `datasets`** lub **NLTK**.

| Dataset | Zadanie | Rozmiar | Źródło |
|---|---|---|---|
| **IMDb** | Analiza sentymentu | 50 000 recenzji | `datasets` / `nltk` |
| **AG News** | Klasyfikacja tematyczna | 120 000 artykułów | `datasets` |
| **SQuAD 2.0** | Question Answering | 150 000 pytań | `datasets` |
| **CoNLL-2003** | Named Entity Recognition | 22 137 zdań | `datasets` |
| **GLUE / SuperGLUE** | Benchmark wielozadaniowy | różne | `datasets` |
| **Common Crawl** | Surowy tekst z internetu | petabajty | commoncrawl.org |
| **Wikipedia dumps** | Encyklopedia | ~4 mld słów (EN) | dumps.wikimedia.org |
| **BookCorpus** | Tekst literacki | ~11 000 książek | `datasets` |
| **OpenWebText** | Tekst z sieci (Reddit) | ~38 GB | `datasets` |
| **C4 (Colossal Clean Crawled Corpus)** | Pretraining | ~750 GB | `datasets` |

In [19]:
# --- Przykład: Pobranie datasetu IMDb z Hugging Face ---
from datasets import load_dataset

# Pobieramy tylko mały fragment (split train, pierwsze 100 przykładów)
imdb = load_dataset("imdb", split="train[:100]")

print(f"Kolumny: {imdb.column_names}")
print(f"Liczba przykładów: {len(imdb)}")
print(f"\nPrzykładowa recenzja (skrócona):")
print(f"  Etykieta: {'pozytywna' if imdb[0]['label'] == 1 else 'negatywna'}")
print(f"  Tekst:    {imdb[0]['text'][:200]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Kolumny: ['text', 'label']
Liczba przykładów: 100

Przykładowa recenzja (skrócona):
  Etykieta: negatywna
  Tekst:    I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...


In [20]:
# --- Przykład: AG News (klasyfikacja wiadomości) ---
ag_news = load_dataset("ag_news", split="train[:100]")

etykiety_ag = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

print(f"Przykłady z AG News:\n")
for i in range(3):
    print(f"[{etykiety_ag[ag_news[i]['label']]}] {ag_news[i]['text'][:120]}...")
    print()

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Przykłady z AG News:

[Business] Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,...

[Business] Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputat...

[Business] Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the ou...



In [21]:
# --- Przykład: Korpusy z NLTK ---
import nltk
nltk.download("gutenberg", quiet=True)
nltk.download("brown", quiet=True)
nltk.download("reuters", quiet=True)

from nltk.corpus import gutenberg, brown, reuters

print("=== Gutenberg (klasyka literatury) ===")
print(f"Dostępne teksty: {gutenberg.fileids()[:5]}")
print(f"Pierwsze 200 znaków 'austen-emma.txt':")
print(gutenberg.raw("austen-emma.txt")[:200])

print("\n=== Brown Corpus (zrównoważony korpus EN) ===")
print(f"Kategorie: {brown.categories()[:8]}")
print(f"Łączna liczba słów: {len(brown.words()):,}")

print("\n=== Reuters (artykuły prasowe) ===")
print(f"Liczba dokumentów: {len(reuters.fileids()):,}")
print(f"Kategorie (pierwsze 10): {reuters.categories()[:10]}")

=== Gutenberg (klasyka literatury) ===
Dostępne teksty: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt']
Pierwsze 200 znaków 'austen-emma.txt':
[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; an

=== Brown Corpus (zrównoważony korpus EN) ===
Kategorie: ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned']
Łączna liczba słów: 1,161,192

=== Reuters (artykuły prasowe) ===
Liczba dokumentów: 10,788
Kategorie (pierwsze 10): ['acq', 'alum', 'barley', 'bop', 'carcass', 'castor-oil', 'cocoa', 'coconut', 'coconut-oil', 'coffee']


---
## 9. Popularne korpusy i datasety — język polski

Ekosystem NLP dla polskiego jest mniejszy niż angielski, ale dynamicznie się rozwija.

| Dataset | Zadanie | Rozmiar | Źródło / ID Hugging Face |
|---|---|---|---|
| **NKJP** (Narodowy Korpus Języka Polskiego) | Uniwersalny | 1,8 mld słów | nkjp.pl |
| **PolEmo 2.0** | Analiza sentymentu | 8 000 recenzji | `clarin-pl/polemo2-official` |
| **KLEJ Benchmark** | Benchmark wielozadaniowy (pol. GLUE) | różne | klejbenchmark.com |
| **Allegro Reviews** | Sentyment / recenzje | 11 588 recenzji | `allegro/allegro-reviews` |
| **PSC (Polish Summaries Corpus)** | Streszczanie | 569 artykułów | `clarin-pl/psc` |
| **Polish Wikipedia** | Tekst encyklopedyczny | ~250 mln słów | dumps.wikimedia.org/plwiki |
| **OSCAR (Polish subset)** | Surowy tekst z sieci | ~47 GB | `oscar-corpus/OSCAR-2301` |
| **CC-100 Polish** | Pretraining | ~26 GB | `cc100` (lang=pl) |
| **PPC (Polish Parliamentary Corpus)** | Korpus parlamentarny | ~0,5 mld słów | clarin-pl.eu |
| **PolQA** | Question Answering | 7 000 pytań | `ipipan/polqa` |

> **KLEJ** (Kompleksowa Lista Ewaluacji Językowych) to polski odpowiednik angielskiego benchmarku GLUE. Zawiera zadania takie jak: analiza sentymentu, wnioskowanie, rozpoznawanie nazw własnych (NER), wykrywanie parafraz i inne.

In [22]:
# --- Przykład: Allegro Reviews ---
allegro = load_dataset("allegro_reviews", split="train[:100]")
print(f"Kolumny: {allegro.column_names}")
print(f"Liczba przykładów: {len(allegro)}\n")

for i in range(3):
    print(f"  Ocena: {allegro[i]['rating']}/5")
    print(f"  Tekst: {allegro[i]['text'][:120]}...")
    print()

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.23M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/349k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/349k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9577 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1006 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1002 [00:00<?, ? examples/s]

Kolumny: ['text', 'rating']
Liczba przykładów: 100

  Ocena: 3.0/5
  Tekst: Jako do ceny dobra. Przyssawka mogłaby być lepsza. Po 2 miesiącach użytkowania musiałem nóżkę z przyssawką rozkręcić i p...

  Ocena: 1.0/5
  Tekst: Na słuchawkę czekałam spory czas a po zadzwonieniu okazało się ,że paczka im się zawieruszyła i w ten sam dzień mieli wy...

  Ocena: 1.0/5
  Tekst: Czajnik na pierwszy rzut oka wygląda ok, ale nie polecam!!! Woda pomimo ciągłego gotowania nieprzyjemnie pachnie, czajni...



In [23]:
# --- Przykład: PolEmo 2.0 (analiza sentymentu PL) ---
polemo = load_dataset("allegro/klej-polemo2-in", split="train[:100]")
print(f"Kolumny: {polemo.column_names}")
print(f"Liczba przykładów: {len(polemo)}\n")

mapowanie = {"__label__z_amb": "ambiwalentny", "__label__z_minus": "negatywny",
              "__label__z_zero": "neutralny", "__label__z_plus": "pozytywny"}

for i in range(3):
    etykieta = polemo[i]["target"]
    print(f"  Sentyment: {mapowanie.get(etykieta, etykieta)}")
    print(f"  Tekst: {polemo[i]['sentence'][:150]}...")
    print()

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5783 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/723 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/722 [00:00<?, ? examples/s]

Kolumny: ['sentence', 'target']
Liczba przykładów: 100

  Sentyment: __label__meta_plus_m
  Tekst: Super lekarz i człowiek przez duże C . Bardzo duże doświadczenie i trafne diagnozy . Wielka cierpliwość do ludzi starszych . Od lat opiekuje się moją ...

  Sentyment: __label__meta_minus_m
  Tekst: Bardzo olewcze podejscie do pacjenta . Przyprowadzajac dziecko z ostra wysypka na calym ciele trwajaca od 2 tygodni Pani doktor stwierdzila ze nie wid...

  Sentyment: __label__meta_amb
  Tekst: Lekarz zalecił mi kurację alternatywną do dotychczasowej , więc jeszcze nie daję najwyższej oceny ( zobaczymy na ile okaże się skuteczna ) . Do Pana d...



In [24]:
# --- Przykład: Pobranie fragmentu polskiej Wikipedii ---
# (Hugging Face hostuje przetworzone wersje)
wiki_pl = load_dataset(
    "wikimedia/wikipedia",
    "20231101.pl",
    split="train[:50]"
)
print(f"Kolumny: {wiki_pl.column_names}")
print(f"\nPrzykładowy artykuł:")
print(f"  Tytuł: {wiki_pl[0]['title']}")
print(f"  Tekst: {wiki_pl[0]['text'][:250]}...")

README.md: 0.00B [00:00, ?B/s]

20231101.pl/train-00000-of-00006.parquet:   0%|          | 0.00/431M [00:00<?, ?B/s]

20231101.pl/train-00001-of-00006.parquet:   0%|          | 0.00/339M [00:00<?, ?B/s]

20231101.pl/train-00002-of-00006.parquet:   0%|          | 0.00/274M [00:00<?, ?B/s]

20231101.pl/train-00003-of-00006.parquet:   0%|          | 0.00/248M [00:00<?, ?B/s]

20231101.pl/train-00004-of-00006.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

20231101.pl/train-00005-of-00006.parquet:   0%|          | 0.00/231M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1587721 [00:00<?, ? examples/s]

Kolumny: ['id', 'url', 'title', 'text']

Przykładowy artykuł:
  Tytuł: AWK
  Tekst: AWK – interpretowany język programowania, którego główną funkcją jest wyszukiwanie i przetwarzanie wzorców w plikach lub strumieniach danych. Jest także nazwą programu początkowo dostępnego dla systemów operacyjnych będących pochodnymi UNIX-a, obecni...


### Przydatne linki do polskich zasobów NLP

- **CLARIN-PL** — [clarin-pl.eu](https://clarin-pl.eu) — główne repozytorium narzędzi i korpusów dla języka polskiego
- **NKJP** — [nkjp.pl](http://nkjp.pl) — Narodowy Korpus Języka Polskiego (1,8 mld słów)
- **KLEJ Benchmark** — [klejbenchmark.com](https://klejbenchmark.com) — benchmark oceny modeli NLP dla polskiego
- **Hugging Face (filtr: `pl`)** — [huggingface.co/datasets?language=pl](https://huggingface.co/datasets?language=pl&sort=downloads) — przeglądaj datasety po języku
- **SpeakLeash** — [speakleash.org](https://speakleash.org) — otwarty projekt budowy dużego korpusu polskiego

---
## 10. Podsumowanie

### Schemat budowy korpusu NLP

```
┌─────────────┐   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│  Pliki TXT  │   │ Pliki CSV   │   │ Pliki DOCX  │   │ Pliki PDF   │
│  (chardet)  │   │ (pandas)    │   │(python-docx)│   │(pdfplumber) │
└──────┬──────┘   └──────┬──────┘   └──────┬──────┘   └──────┬──────┘
       │                 │                  │                  │
       └────────┬────────┴──────────┬───────┴──────────────────┘
                │                   │
                ▼                   ▼
        ┌───────────────────────────────────┐
        │    Ujednolicony korpus (DataFrame) │
        │   plik | format | tekst | ...      │
        └───────────────┬───────────────────┘
                        │
                        ▼
        ┌───────────────────────────────────┐
        │     Czyszczenie i normalizacja    │
        │ (Unicode NFC, białe znaki, lower) │
        └───────────────┬───────────────────┘
                        │
                        ▼
        ┌───────────────────────────────────┐
        │      Gotowy korpus do analizy     │
        │    (zapis: JSONL / CSV / Parquet) │
        └───────────────────────────────────┘
```

### Kluczowe wnioski

1. **Kodowanie ma znaczenie** — szczególnie dla polskiego. Zawsze sprawdzaj i normalizuj (UTF-8 + NFC).
2. **Różne formaty = różne narzędzia** — `chardet` dla TXT, `pandas` dla CSV, `python-docx` dla Word, `pdfplumber` dla PDF.
3. **Zachowaj metadane** — informacja o źródle, formacie i strukturze dokumentu przydaje się w dalszych analizach.
4. **Korzystaj z istniejących datasetów** — Hugging Face `datasets` to najprostszy sposób na pobranie gotowych korpusów.
5. **Polski NLP rośnie** — projekty takie jak KLEJ, CLARIN-PL i SpeakLeash dostarczają coraz więcej zasobów.
